# Self-Ask with Web Search (Groq + Tavily)

**Author:** Ibrahim   
**Environment:** Google Colab / Python 3

## Overview
This notebook implements a **self-ask agent** that can search the web to answer questions. Given a query, the LLM decides whether it needs external information. If yes, it generates a search query, retrieves results via Tavily API, and then produces the final answer. This pattern is used in many production AI assistants.

## What You Will Build
- A function that uses Groq to decide if a web search is needed.
- Integration with Tavily search API.
- Automatic execution: think → search (if needed) → answer.
- Interactive loop to ask any factual or current question.

## Requirements
- **Groq API key** (free from [console.groq.com](https://console.groq.com))
- **Tavily API key** (free from [tavily.com](https://tavily.com))

---

**© 2026 Ibrahim – Self-ask agent with web search.**

### Install & Imports

In [1]:
!pip install -q groq tavily-python

import os
import json
from getpass import getpass
from groq import Groq
from tavily import TavilyClient

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.9 MB/s eta 0:00:00


### API Keys & Clients

In [2]:
GROQ_API_KEY = getpass("Enter your Groq API key: ")
TAVILY_API_KEY = getpass("Enter your Tavily API key: ")

groq_client = Groq(api_key=GROQ_API_KEY)
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
MODEL = "llama-3.3-70b-versatile"
print("Clients ready.")

Enter your Groq API key: ··········
Enter your Tavily API key: ··········
Clients ready.


### Self-Ask Function

In [3]:
def self_ask(query, max_search=1):
    """
    Decide if web search is needed, search if yes, then answer.
    Returns final answer as string.
    """
    # First, ask the LLM if a search is needed
    decision_prompt = f"""You are a helpful assistant. Decide if you need to search the web to answer the user's question.
If you already know the answer from your training data, output "ANSWER: <your answer>".
If you need fresh or specific information, output "SEARCH: <search query>".

User question: {query}

Output (ANSWER or SEARCH):"""

    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": decision_prompt}],
        temperature=0,
        max_tokens=128
    )
    decision = response.choices[0].message.content.strip()

    if decision.startswith("ANSWER:"):
        answer = decision.replace("ANSWER:", "").strip()
        return answer, False

    elif decision.startswith("SEARCH:"):
        search_query = decision.replace("SEARCH:", "").strip()
        print(f"🔍 Searching for: {search_query}")
        try:
            search_result = tavily_client.search(search_query, max_results=3)
            snippets = [r['content'] for r in search_result.get('results', [])]
            context = "\n\n".join(snippets)
        except Exception as e:
            context = f"Search failed: {e}"

        final_prompt = f"""Use the following search results to answer the user's question. If the results are insufficient, say so.

User question: {query}
Search results:
{context}

Answer (concise and factual):"""
        response2 = groq_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": final_prompt}],
            temperature=0,
            max_tokens=512
        )
        answer = response2.choices[0].message.content.strip()
        return answer, True
    else:
        return "Unable to parse decision. Please rephrase.", False

### Test with Examples

In [4]:
test_queries = [
    "What is the current population of Japan?",
    "Who won the FIFA World Cup in 2018?",
    "Explain the theory of relativity."
]

for q in test_queries:
    print(f"\n {q}")
    ans, searched = self_ask(q)
    if searched:
        print(" Used web search")
    else:
        print(" Used internal knowledge")
    print(f" {ans}\n")


 What is the current population of Japan?
🔍 Searching for: current population of Japan
 Used web search
 The current population of Japan is approximately 122.5 million people, with the 2026 mid-year estimate being 122,427,731.


 Who won the FIFA World Cup in 2018?
 Used internal knowledge
 France won the FIFA World Cup in 2018.


 Explain the theory of relativity.
 Used internal knowledge
 The theory of relativity, developed by Albert Einstein, is a fundamental concept in modern physics that describes the nature of space and time. It consists of two main components: special relativity and general relativity. Special relativity posits that the laws of physics are the same for all observers in uniform motion relative to one another, and that the speed of light is always constant, regardless of the motion of the observer. This theory challenged the long-held notion of absolute time and space, and introduced the concept of time dilation and length contraction. General relativity builds u

### Interactive Loop

In [5]:
print("\nSelf-Ask Agent Ready. Type 'exit' to quit.\n")
while True:
    q = input("You: ").strip()
    if q.lower() == "exit":
        break
    if not q:
        continue
    ans, _ = self_ask(q)
    print(f"Bot: {ans}\n")


Self-Ask Agent Ready. Type 'exit' to quit.

You: What is the current population of India?
🔍 Searching for: current population of India
Bot: The current population of India is approximately 1.474 billion as of May 10, 2026.

You: Who wrote the novel 'Pride and Prejudice'?
Bot: Jane Austen

You: What is the weather like in London today?
🔍 Searching for: London weather today
Bot: The current weather in London is partly cloudy with a temperature of 10.3°C (50.5°F) and a feels-like temperature of 8.8°C (47.8°F). The wind is 7.2 mph (11.5 kph) from the NNW direction, and the humidity is 66%. There is a 100% chance of rain.

You: exit


### Final Summary

In [6]:
print("Self-Ask with Web Search - COMPLETED")
print("Author: Ibrahim")
print(" LLM decides when to search the web.")
print(" Uses Tavily for real‑time information.")
print(" Works for factual, current, or niche questions.")

Self-Ask with Web Search - COMPLETED
Author: Ibrahim
 LLM decides when to search the web.
 Uses Tavily for real‑time information.
 Works for factual, current, or niche questions.
